# Circuit Breaker Pattern | Agent Safety & Resilience

In [1]:
# Circuit Breaker for Agent Tool Calls
import time
from typing import Callable, Any
from dataclasses import dataclass

In [2]:
@dataclass
class CircuitBreaker:
    failure_threshold: int = 3
    recovery_timeout: float = 30.0  # seconds
    _failure_count: int = 0
    _state: str = "closed"  # closed, open, half_open
    _last_failure_time: float = 0.0

    def call(self, func: Callable, *args, fallback: Any = None, **kwargs) -> Any:
        """Execute a function through the circuit breaker."""
        if self._state == "open":
            if time.time() - self._last_failure_time > self.recovery_timeout:
                self._state = "half_open"
            else:
                print(f"Circuit OPEN: returning fallback")
                return fallback

        try:
            result = func(*args, **kwargs)
            if self._state == "half_open":
                self._state = "closed"
                self._failure_count = 0
                print("Circuit recovered: CLOSED")
            return result
        except Exception as e:
            self._failure_count += 1
            self._last_failure_time = time.time()
            if self._failure_count >= self.failure_threshold:
                self._state = "open"
                print(f"Circuit OPENED after {self._failure_count} failures")
            raise

    @property
    def state(self) -> str:
        return self._state

In [3]:
# Usage with an agent's tool call
cb = CircuitBreaker(failure_threshold=3, recovery_timeout=10)

def unreliable_api_call(query: str) -> str:
    """Simulates an API that might fail."""
    import random
    if random.random() < 0.7:  # 70% failure rate for demo
        raise ConnectionError("API unavailable")
    return f"Result for: {query}"

# Try multiple calls through the circuit breaker
for i in range(5):
    try:
        result = cb.call(unreliable_api_call, f"query-{i}",
                         fallback="Service temporarily unavailable. Using cached response.")
        print(f"Call {i}: {result}")
    except ConnectionError:
        print(f"Call {i}: Failed (circuit state: {cb.state})")

Call 0: Failed (circuit state: closed)
Call 1: Failed (circuit state: closed)
Call 2: Result for: query-2
Circuit OPENED after 3 failures
Call 3: Failed (circuit state: open)
Circuit OPEN: returning fallback
Call 4: Service temporarily unavailable. Using cached response.
